# Inspection Bay — Data Quality Check

Put your data up on the lift and look for problems before you start working with it.

See the [Inspection README](README.md) for full documentation.

In [ ]:
# ============================================================
# SALON CONFIG — edit the line below to point to your CSV file
# ============================================================
import pandas as pd
from pathlib import Path

PARKING_BAY = Path("../../parking_bay")
CSV_FILE = "your_file.csv"  # <-- change this to your file name

df = pd.read_csv(PARKING_BAY / CSV_FILE)
print(f"Loaded: {CSV_FILE}  |  Shape: {df.shape}")

## Missing Values

How many nulls are in each column, and what percentage of the column does that represent?

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

missing = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct
})
missing = missing[missing["null_count"] > 0].sort_values("null_pct", ascending=False)

if missing.empty:
    print("No missing values found.")
else:
    display(missing)

## Duplicate Rows

Are there any rows that are exact duplicates of another row?

In [ ]:
dupe_count = df.duplicated().sum()
print(f"Duplicate rows: {dupe_count} ({dupe_count / len(df) * 100:.2f}% of total)")

if dupe_count > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()).head(20))

## Data Type Audit

Are the columns the right types? Numbers stored as strings, dates stored as objects?

In [ ]:
type_audit = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null_count": df.notnull().sum(),
    "null_count": df.isnull().sum(),
    "unique_values": df.nunique()
})
display(type_audit)

## Unique Value Counts

How many distinct values does each column have? Low-cardinality columns are likely categorical.

In [ ]:
unique_counts = df.nunique().sort_values()
for col, count in unique_counts.items():
    print(f"  {col}: {count} unique values")

## Outliers — Numeric Columns (IQR Method)

Flags values that fall below Q1 - 1.5*IQR or above Q3 + 1.5*IQR. These may be errors or genuinely extreme values.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if not outliers.empty:
        print(f"\n{col}: {len(outliers)} outlier(s)  |  bounds: [{lower:.2f}, {upper:.2f}]")
        display(outliers[[col]].describe())
    else:
        print(f"{col}: no outliers")

## Value Counts — String / Categorical Columns

What are the most common values in each text column? Useful for spotting typos, inconsistent casing, or unexpected categories.

In [ ]:
string_cols = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]

for col in string_cols:
    print(f"\n--- {col} ---")
    display(df[col].value_counts().head(10))

---
## Script Generator

Delete any blocks above that you don't want in the output, then run this cell.
A clean `.py` script will be saved to the `scripts/` folder.

In [ ]:
# ============================================================
# SCRIPT GENERATOR
# ============================================================
import nbformat
from pathlib import Path

NOTEBOOK_NAME = "inspection.ipynb"

nb = nbformat.read(NOTEBOOK_NAME, as_version=4)

code_cells = [
    cell for cell in nb.cells
    if cell.cell_type == "code"
    and cell.source.strip()
    and "# SCRIPT GENERATOR" not in cell.source
]

script = "\n\n".join(cell.source for cell in code_cells)

output_dir = Path("../../scripts")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / f"{Path(NOTEBOOK_NAME).stem}_script.py"
output_file.write_text(script, encoding="utf-8")

print(f"Script saved to: {output_file.resolve()}")